# 03. NLP Sentiment Analysis

This notebook applies Natural Language Processing (NLP) over the raw Google Maps reviews (`gmaps_reviews.csv`). 
It uses NLTK's VADER sentiment analyzer to compute a polarity score for each text snippet, then aggregates this sentiment up to the restaurant level to be fed into the final scoring pipeline.

In [1]:
import pandas as pd
import numpy as np
import os
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import ssl
import warnings
warnings.filterwarnings('ignore')

## 1. Environment & Setup

Download VADER lexicon and prepare directories.

In [2]:
# Override SSL verification for NLTK downnload in strict corporate networks
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('vader_lexicon', quiet=True)

OUTPUT_DIR = "Output_Data"
INPUT_FILE = "../gmaps_reviews.csv"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "master_nlp_data.csv")

## 2. Process Reviews

In [3]:
print(f"Loading {INPUT_FILE}...")
df_rev = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df_rev)} reviews.")

df_clean = df_rev.dropna(subset=['text_snippet']).copy()
print(f"Kept {len(df_clean)} non-null text snippets.")

sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    return sia.polarity_scores(str(text))['compound']

print("Applying VADER sentiment analysis...")
df_clean['compound_polarity'] = df_clean['text_snippet'].apply(get_sentiment)

df_clean.head()

Loading ../gmaps_reviews.csv...
Loaded 8711 reviews.
Kept 8581 non-null text snippets.
Applying VADER sentiment analysis...


,restaurant_id,rating,relative_date,text_snippet,scraped_at,review_date,compound_polarity
0,168,5.0,a week ago,I came quite early at 5:30 p.m. so it was very...,2026-02-08T02:54:17.594257,2026-02-01,0.8809
1,168,5.0,a month ago,"The pasta was cooked to perfection, with a che...",2026-02-08T02:54:17.616458,2026-01-09,0.9524
2,168,5.0,3 months ago,"Everything was so nice, the food was really go...",2026-02-08T02:54:17.638212,2025-11-10,0.9608
3,168,5.0,3 months ago,Beautiful rooftop views of Bkk night lights. G...,2026-02-08T02:54:17.660223,2025-11-10,0.8979
4,168,5.0,2 months ago,"Great, intimate dining experience. Sunset view...",2026-02-08T02:54:17.682517,2025-12-10,0.8934


## 3. Aggregate by Restaurant

Average the polarity score, compute variance (consistency of service), and convert to a 0-100 scale.

In [4]:
nlp_agg = df_clean.groupby('restaurant_id').agg(
    raw_average_sentiment=('compound_polarity', 'mean'),
    sentiment_variance=('compound_polarity', 'var'),
    nlp_review_count=('compound_polarity', 'count')
).reset_index()

nlp_agg['sentiment_variance'] = nlp_agg['sentiment_variance'].fillna(0)

# VADER compound score ranges from -1 to 1. 
# Map this securely to a 0 to 100 positive scale to match other metrics.
nlp_agg['sentiment_score'] = ((nlp_agg['raw_average_sentiment'] + 1) / 2) * 100

nlp_agg.to_csv(OUTPUT_FILE, index=False)

print(f"✅ NLP Analysis Complete! Aggregated {len(df_clean)} reviews into {len(nlp_agg)} unique restaurants.")
print(f"Saved to {OUTPUT_FILE}")
nlp_agg.head()

✅ NLP Analysis Complete! Aggregated 8581 reviews into 325 unique restaurants.
Saved to Output_Data\master_nlp_data.csv


,restaurant_id,raw_average_sentiment,sentiment_variance,nlp_review_count,sentiment_score
0,33,0.690793,0.123127,30,84.539667
1,168,0.820500,0.031579,20,91.025000
2,222,0.678475,0.115896,20,83.923750
3,423,0.688165,0.143000,20,84.408250
4,425,0.698107,0.145517,30,84.905333
